# 金融人工智慧 — PyCaret 的 AutoML 實作（個人作業）

**學號**：15999511　**姓名**：栗嘉良　**系級**：財務工程與精算數學系 三年級

**選用資料集**：UCI *Bank Marketing*（銀行電話行銷／客戶是否認購定期存款，二元分類）
**對應教程**：PyCaret 101（分類 AutoML）+ 102（ydata-profiling 資料品質報告）
**本次作業四大任務**：
- 任務 1（101）：從 Colab 外部載入「非 credit」的二元分類資料集，並跑完 101 教程所有分析步驟
- 任務 2（101）：把教程中對資料集的「說明文字區塊」改成新資料集的問題與欄位說明
- 任務 3（101）：針對 Feature Importance 段落，解釋重要變量（XAI）
- 任務 4（102）：用 ydata-profiling 套件產製自己資料的「資料品質報告」

> 執行方式：在 Google Colab 開啟本筆記本 → 依序「執行所有儲存格」。首次會安裝套件（約 2–3 分鐘）。


## 任務 1：從 Colab 外部載入資料集並建立實驗環境

作業要求「選擇任何一個二元分類資料集（除了教程的 credit 之外），從 Colab 外部載入」。
我選 **UCI Bank Marketing**：預測銀行客戶在電話行銷中是否會認購定期存款（term deposit）。
原因：① 它是金融場景的二元分類；② 資料筆數大（45,211 筆）、欄位豐富；③ 公開且可用網址直接下載，符合「外部載入」。


In [ ]:
# 安裝本作業需要的套件（Colab 第一次執行約需 2-3 分鐘，之後會快很多）
!pip install -q pycaret
!pip install -q ydata-profiling
!pip install -q shap   # interpret_model(SHAP 解釋) 需要的套件


In [ ]:
import pandas as pd

# 任務 1：從 Colab「外部」載入資料集
# 做法 A（本筆記本採用）：用 wget 從 UCI 官網下載 zip 再解壓縮
# 做法 B（你也可以）：點 Colab 左側 ▸ 上傳，把 csv 傳上去後改為 pd.read_csv('你的檔名.csv')
!wget -q "https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank.zip" -O bank.zip
!unzip -o -q bank.zip

# 讀取資料（bank-full.csv 以分號 ';' 分隔，不是常見的逗號）
data = pd.read_csv('bank-full.csv', sep=';')
print('資料筆數與欄位數：', data.shape)
data.head()


## 任務 2：資料集說明（修改自教程的 credit 文字區塊）

### 這個資料在問什麼問題？
銀行想預測：**某位客戶在這一輪電話行銷中，最後會不會認購定期存款？**
這是金融業典型的「行銷回應預測（response modeling）」任務。若能提前知道誰較可能認購，
銀行就能把有限的通話人力與成本集中在高潛力客戶上——本質上是用機器學習做「資源配置決策」。

### 標籤（要預測的目標）
- `y`：客戶是否認購定期存款 → `yes` / `no`（二元分類標籤）

### 欄位（特徵）說明
| 欄位 | 含義 | 類型 |
|---|---|---|
| age | 年齡 | 數值 |
| job | 職業類型（如 management / technician / student …） | 類別 |
| marital | 婚姻狀態（married / single / divorced） | 類別 |
| education | 教育程度（primary / secondary / tertiary） | 類別 |
| default | 是否曾有信用違約 | 類別（yes/no） |
| balance | 每年帳戶餘額（歐元） | 數值 |
| housing | 是否有房屋貸款 | 類別（yes/no） |
| loan | 是否有個人貸款 | 類別（yes/no） |
| contact | 聯絡方式（cellular / telephone / unknown） | 類別 |
| day | 最後一次聯絡的「月份中的第幾天」 | 數值 |
| month | 最後一次聯絡的月份 | 類別 |
| duration | 最後一次通話長度（秒） | 數值 |
| campaign | 本次活動中對該客戶的聯絡次數 | 數值 |
| pdays | 距離上一次聯絡該客戶的天數（-1 = 從未聯絡） | 數值 |
| previous | 本次活動「之前」對該客戶的聯絡次數 | 數值 |
| poutcome | 上一次行銷活動的結果（success / failure / other / unknown） | 類別 |

> ⚠️ 重要資料科學觀念：`duration`（通話秒數）雖然最強，但它是**通話結束後才會知道**的數字，
> 等於「答案已經出來才知道的線索」，在真實上線模型裡不能使用（資料洩漏 data leakage）。本作業先用它跑通流程，
> 但下結論時要特別標註這點（見任務 3）。


## PyCaret 101 教程步驟：建立 AutoML 實驗

下面依序對應教程：初始化環境 → 比較所有模型 → 調參 → 視覺化 → 預測 → 定稿存檔。


In [ ]:
from pycaret.classification import *

# 初始化實驗環境：指定目標欄位 y（yes/no 二元分類）
# session_id=123 讓結果可重現；silent=True 避免互動詢問；html=False 不另開網頁
# n_jobs=-1：用 Colab 所有 CPU 核心平行訓練，大幅加速（這是前面跑很久的主因）
setup(data, target='y', session_id=123, silent=True, html=False, n_jobs=-1)


### ⚠️ 本資料「類別不平衡」的提醒
目標 `y` 中 `no` 佔 **88.3%**、`yes` 只佔 11.7%。這會讓「準確率」失真——
因為「全部猜不認購」就能拿 88.3% 準確率，卻一個潛在客戶都抓不到。
因此下面改以 **AUC** 作為挑選模型的指標（AUC 不受類別比例影響，更能反映真正的區分能力）。


In [ ]:
# 步驟 1：比較所有模型
# compare_models() 會自動訓練多種分類模型、用交叉驗證評分並回傳最好的模型
# 因資料不平衡，改用 sort='AUC' 挑選（而非預設的 Accuracy）
# 加速設定：只挑有代表性的快模型 + 5 折 + n_jobs=-1 平行訓練，避免 Colab 跑幾十分鐘
# （想跑更完整可把 include 拿掉，改回比較全部模型，但會慢很多）
FAST_MODELS = ['lr', 'dt', 'rf', 'gbc', 'lightgbm', 'xgboost', 'catboost', 'ada']
best = compare_models(sort='AUC', include=FAST_MODELS, fold=5, n_jobs=-1)
best


In [ ]:
# 步驟 2：對最好的模型做「超參數調整」，讓它更強
tuned = tune_model(best, choose_better=True)
tuned


In [ ]:
# 步驟 3：視覺化評估模型
plot_model(tuned, plot='auc')              # AUC 曲線（越大越好，上限 1）
plot_model(tuned, plot='confusion_matrix') # 混淆矩陣（看真陽/偽陽/偽陰/真陰）
plot_model(tuned, plot='feature_importance') # 特徵重要性（哪些欄位最影響預測）
plot_model(tuned, plot='class_report')     # 精確率/召回率/F1 報告


In [ ]:
# 步驟 4：在「沒看過的測試資料」上預測，檢查泛化能力
prediction = predict_model(tuned)
prediction.head()


In [ ]:
# 步驟 5：用全部資料重新訓練最終模型並存檔
final_model = finalize_model(tuned)
save_model(final_model, 'bank_marketing_pycaret_model')


## 任務 3：Feature Importance 解釋（XAI）

下面先用 PyCaret 內建的 SHAP 解釋器看「每個特徵如何影響預測」，再用人話解釋最重要的幾個變量。


In [ ]:
# 用 SHAP 解釋模型：summary plot 顯示每個特徵對「認購機率」的正/負影響方向與強度
# 說明：紅色=該特徵數值大，藍色=數值小；點越往右代表越推高「認購」機率
interpret_model(tuned, plot='summary')


### 對重要變量的解釋（人話版）

以本地用相同資料（random_state=123）跑出的結果為例（你的 Colab 數字會非常接近）：

- **duration（通話秒數）— 重要性最高（約 0.27）**：通話越久，客戶越可能認購。
  但如前面所警示，這是「通話結束後才知」的變數，屬資料洩漏，**真實決策模型應剔除**。
- **balance（帳戶餘額，約 0.10）**：餘額越高，認購能力越強、意願通常越高。這是合理且可用的金融特徵。
- **age（年齡，約 0.097）**：年齡與認購行為有關，但非線性（中年客群往往較穩健）。
- **month（聯絡月份，整體約 0.10）**：行銷成果有明顯季節性（如 mar / oct / dec 往往較好），反映銀行推廣節奏。
- **poutcome（上一次活動結果，約 0.064）**：上次就 success 的客戶，這次再認購機率顯著較高——這是最有「行為預測力」的特徵之一。
- **job / housing / loan / default** 等也都有貢獻，但相對次要。

> 小結：模型能達到約 **90% 準確率、AUC 約 0.93**，主要是因為 `duration` 這個「類似答案的變數」。
> 若移除 `duration` 再訓練，準確率會明顯下降——這正好示範了「高準確率 ≠ 好模型」，
> 必須配合對資料生成邏輯的理解（能力圈）才不會被數字騙。


## 任務 4：ydata-profiling 資料品質報告（102 教程）

自動產生一份互動式資料健檢報告：缺失值、分佈、相關性、重複值等一目了然。


In [ ]:
# 任務 4：產製資料品質報告
from ydata_profiling import ProfileReport

profile = ProfileReport(data, title='Bank Marketing 資料品質報告', explorative=True)
profile.to_notebook_iframe()          # 在 Colab 內直接顯示互動報告
profile.to_file('bank_profile.html')  # 存成 html 檔，方便隨作業繳交


## 進階實證：duration 的資料洩漏（建議放在最後執行）

下面用一個「對照實驗」把前面講的資料洩漏變成看得見的數字：
把 `duration` 拿掉 → 重新初始化環境 → 重新比較模型，看 AUC 掉多少。
> 注意：這段會「重新初始化實驗環境」並覆蓋前面的 setup，請放在最後執行；若不想多等幾分鐘也可跳過（結論不變）。


In [ ]:
# 進階實證：移除 duration 後再建模，觀察 AUC 變化
# 1) 刪除「通話秒數」這個會洩漏答案的欄位
data_wo = data.drop(columns=['duration'])

# 2) 重新初始化實驗環境（此行會覆蓋前面的 setup）
setup(data_wo, target='y', session_id=123, silent=True, html=False, n_jobs=-1)

# 3) 重新比較模型，同樣用 AUC 挑選（同樣用加速設定）
best_wo = compare_models(sort='AUC', include=FAST_MODELS, fold=5, n_jobs=-1)
best_wo


### 實證結果怎麼看？
對照前面「有 duration」的 `best`（AUC 約 0.92~0.93）與這裡「無 duration」的 `best_wo`（AUC 通常掉到約 0.80 附近），
差距就是 `duration` 帶來的「虛高」。這證明了：模型表現好，往往只是因為它偷看了答案。
**真正能上線的模型，必須剔除 duration**，改用 balance / age / month / poutcome 等「通話前就已知」的特徵來預測。


## 結論 / 心得（繳交前請補上自己的話）

- 本作業用 PyCaret 在 Bank Marketing 上跑通了完整的 AutoML 流程（compare → tune → evaluate → finalize）。
- 透過 Feature Importance 與 SHAP，理解了哪些變量驅動「認購」預測。
- 最重要的收穫：**模型表現好，不一定代表模型可用**——`duration` 的資料洩漏提醒我們要先懂資料再信數字。
- 用 ydata-profiling 快速完成資料健檢，確認資料無明顯缺失/異常後才建模。

> 繳交方式：在 Colab 選「檔案 ▸ 下載 ▸ 下載 .ipynb」，與 `bank_profile.html` 一起壓成 zip 上傳至 TronClass 作業區。
